# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string in Step 0, then run each cell in order. You will load RAG chunks, create vector and BM25 search indexes, retrieve context with vector and hybrid search, and assemble a grounded prompt for a chat model.

The final cell prints the prompt that your application would send to Azure OpenAI or Azure AI Foundry. The lab keeps the model call out of scope so the database retrieval mechanics are visible.


## Step 0: Connect to Azure DocumentDB

This cell restores the MongoDB driver, accepts your connection string, and opens `docdbworkshop.rag_chunks`.

In [ ]:
#r "nuget: MongoDB.Driver, 3.4.0"
using MongoDB.Bson;
using MongoDB.Driver;
using System.Linq;

var connectionString = Environment.GetEnvironmentVariable("DOCUMENTDB_CONNECTION_STRING") ?? "<paste-your-azure-documentdb-connection-string-here>";
if (connectionString.Contains("<paste")) throw new Exception("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
var client = new MongoClient(connectionString);
var db = client.GetDatabase("docdbworkshop");
var chunks = db.GetCollection<BsonDocument>("rag_chunks");
db.RunCommand<BsonDocument>(new BsonDocument("ping", 1))

## Step 1: Load RAG chunks

Each chunk stores source text, metadata, and an embedding together.

In [ ]:
chunks.DeleteMany(FilterDefinition<BsonDocument>.Empty);
chunks.InsertMany(new[] {
    new BsonDocument { {"_id","rag-001"}, {"title","Vector search"}, {"chunk","Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity."}, {"url","module-4-search"}, {"embedding",new BsonArray{0.92,0.80,0.18}} },
    new BsonDocument { {"_id","rag-002"}, {"title","Full-text search"}, {"chunk","Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches."}, {"url","module-4-search"}, {"embedding",new BsonArray{0.20,0.12,0.94}} },
    new BsonDocument { {"_id","rag-003"}, {"title","Hybrid search"}, {"chunk","Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion."}, {"url","module-4-search"}, {"embedding",new BsonArray{0.76,0.70,0.42}} },
    new BsonDocument { {"_id","rag-004"}, {"title","Grounded generation"}, {"chunk","A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data."}, {"url","module-5-rag"}, {"embedding",new BsonArray{0.84,0.73,0.34}} }
});
chunks.CountDocuments(FilterDefinition<BsonDocument>.Empty)

## Step 2: Create retrieval indexes

Create a DiskANN vector index and a BM25 full-text search index.

In [ ]:
db.RunCommand<BsonDocument>(new BsonDocument{{"createIndexes","rag_chunks"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_chunk_embedding_diskann"},{"key",new BsonDocument("embedding","cosmosSearch")},{"cosmosSearchOptions",new BsonDocument{{"kind","vector-diskann"},{"dimensions",3},{"similarity","COS"},{"maxDegree",32},{"lBuild",64}}}}}}});
db.RunCommand<BsonDocument>(new BsonDocument{{"createSearchIndexes","rag_chunks"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_chunk_fts"},{"definition",new BsonDocument("mappings",new BsonDocument{{"dynamic",false},{"fields",new BsonDocument("chunk",new BsonDocument("type","string"))}})}}}}});

## Step 3: Retrieve vector context

The query vector returns the most semantically similar chunks.

In [ ]:
var question = "How does DocumentDB retrieve context for RAG?";
var questionVector = new BsonArray {0.83, 0.74, 0.33};
var vectorContext = chunks.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument("cosmosSearch", new BsonDocument{{"path","embedding"},{"vector",questionVector},{"k",3}})),
    new BsonDocument("$project", new BsonDocument{{"_id",1},{"title",1},{"chunk",1},{"url",1},{"score",new BsonDocument("$meta","searchScore")}})
}).ToList();
vectorContext

## Step 4: Build the grounded prompt

The retrieved chunks become the context block for the model prompt.

In [ ]:
var contextBlock = string.Join("

", vectorContext.Select((d, i) => $"[{i + 1}] {d["title"]}
{d["chunk"]}
Source: {d["url"]}"));
var groundedPrompt = $"""You are a helpful assistant for an Azure DocumentDB workshop.
Answer using only the context below. If the answer is missing, say you do not know.

<context>
{contextBlock}
</context>

Question: {question}""";
groundedPrompt